# Travail demandé — Étude et expérimentation de 29 algorithmes de Machine Learning (CC Complet)
**Public :** étudiants de licence avancée / master — **Support :** ce notebook Google Colab complété.
Ce document contient l'intégralité des réponses aux consignes (Partie A et Partie B) avec l'utilisation de données réelles pour l'évaluation.


In [ ]:
!pip -q install scikit-learn xgboost umap-learn gymnasium pandas matplotlib seaborn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression, make_classification, make_blobs, load_wine, fetch_openml, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, f1_score, silhouette_score, classification_report
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline

GRAINE = 42
# Données synthétiques pour la Partie A (démonstrations Q4)
X_reg, y_reg = make_regression(n_samples=280, n_features=5, noise=13, random_state=GRAINE)
X_cls, y_cls = make_classification(n_samples=400, n_features=6, n_informative=4, n_redundant=0, random_state=GRAINE)
X_clu, etiquettes_reference = make_blobs(n_samples=240, centers=3, cluster_std=0.85, random_state=GRAINE)

Xr_ent, Xr_test, yr_ent, yr_test = train_test_split(X_reg, y_reg, test_size=.25, random_state=GRAINE)
Xc_ent, Xc_test, yc_ent, yc_test = train_test_split(X_cls, y_cls, test_size=.25, stratify=y_cls, random_state=GRAINE)
print('Régression:', X_reg.shape, 'Classification:', X_cls.shape)


## 1. Régression linéaire


### Présentation
**Nom :** Régression linéaire ordinaire (Ordinary Least Squares - OLS).
**Famille :** Régression supervisée.
**Intuition :** Ajuster un hyperplan qui minimise la somme des carrés des erreurs (résidus) entre les prédictions et les valeurs réelles. Entrées: X -> Apprentissage: minimisation MCO -> Prédiction: y_hat = X*beta.
**Formule :** $\hat{y}_i = \beta_0 + \sum_{j=1}^p \beta_j x_{ij}$. L'objectif est de minimiser $J(\beta) = \frac{1}{2n} \sum (y_i - \hat{y}_i)^2$. $\beta_j$ sont les paramètres appris, pas d'hyperparamètres stricts (hors fit_intercept).
**Avantages :** Rapide, interprétabilité directe des coefficients.
**Limites :** Sensible aux valeurs atypiques, incapable de modéliser des relations non linéaires.


### Schéma explicatif
```python
plt.scatter(Xr_ent[:,0], yr_ent, alpha=0.5)
plt.plot(Xr_ent[:,0], modele.predict(Xr_ent), color='red')
plt.title('Schéma: Droite d\'ajustement minimisant les résidus')
plt.show()
```
*Commentaire de lecture :* La droite rouge représente l'hyperplan optimal qui passe au plus près du nuage de points, minimisant la distance verticale (résidus).


### Contextes d'utilisation
**Contextes :**
- *Gestion :* Prévision du chiffre d'affaires selon les dépenses publicitaires.
- *Logistique :* Estimation du coût de transport selon la distance.
**Prérequis :** Données quantitatives, absence de multicolinéarité parfaite.
**Peu approprié :** Séries temporelles très volatiles ou relations purement non linéaires (ex: cycle de vie d'un produit).
**Métriques :** MAE, RMSE, R².


### Analyse de l'expérience Python


In [ ]:
from sklearn.linear_model import LinearRegression
modele=LinearRegression().fit(Xr_ent,yr_ent)
print('MAE:',mean_absolute_error(yr_test,modele.predict(Xr_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 2. Ridge


### Présentation
**Nom :** Régression Ridge (Régularisation L2).
**Famille :** Régression supervisée.
**Intuition :** Similaire à la régression linéaire, mais ajoute une pénalité sur la taille des coefficients pour éviter le surapprentissage. 
**Formule :** $J(\beta) = \frac{1}{n} \|y - X\beta\|_2^2 + \alpha \sum \beta_j^2$. $\alpha$ est l'hyperparamètre de régularisation, $\beta$ les paramètres appris.
**Avantages :** Gère bien la multicolinéarité, réduit la variance.
**Limites :** N'annule pas les coefficients (pas de sélection de variables).


### Schéma explicatif
```python
# Schéma conceptuel : La régularisation compresse les coefficients vers zéro.
```
*Commentaire :* Graphiquement, la contrainte L2 forme un cercle (ou hypersphère) autour de l'origine limitant la norme des poids de la régression.


### Contextes d'utilisation
**Contextes :**
- *Finance :* Scoring de crédit avec de nombreux ratios financiers corrélés.
- *Marketing :* Modélisation de ventes avec des variables fortement dépendantes.
**Prérequis :** Standardisation impérative des données.
**Peu approprié :** Quand on souhaite éliminer explicitement des variables (modèle parcimonieux).
**Métriques :** MAE, RMSE, R².


### Analyse de l'expérience Python


In [ ]:
from sklearn.linear_model import Ridge
modele=make_pipeline(StandardScaler(),Ridge(alpha=1.0)).fit(Xr_ent,yr_ent)
print('MAE:',mean_absolute_error(yr_test,modele.predict(Xr_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 3. Lasso


### Présentation
**Nom :** Régression Lasso (Least Absolute Shrinkage and Selection Operator - L1).
**Famille :** Régression supervisée.
**Intuition :** Pénalise la somme des valeurs absolues des coefficients, forçant certains à devenir exactement nuls (sélection de variables).
**Formule :** $J(\beta) = \frac{1}{2n} \|y - X\beta\|_2^2 + \alpha \sum |\beta_j|$. $\alpha$ est l'hyperparamètre.
**Avantages :** Produit des modèles parcimonieux, sélection automatique des features.
**Limites :** Choix arbitraire parmi des variables très corrélées, limité si p > n.


### Schéma explicatif
```python
# Schéma conceptuel : Contrainte L1 (losange)
```
*Commentaire :* La contrainte géométrique du Lasso forme un losange. Les tangences se font souvent sur les axes, annulant ainsi des coefficients spécifiques.


### Contextes d'utilisation
**Contextes :**
- *Finance :* Sélection d'indicateurs macro-économiques parmi des centaines pour prévoir l'inflation.
- *Santé/Gestion :* Identifier les 5 facteurs clés impactant la durée d'hospitalisation.
**Prérequis :** Standardisation obligatoire.
**Peu approprié :** Quand toutes les variables ont un impact dense et utile.
**Métriques :** R², RMSE.


### Analyse de l'expérience Python


In [ ]:
from sklearn.linear_model import Lasso
modele=make_pipeline(StandardScaler(),Lasso(alpha=.1,max_iter=10000)).fit(Xr_ent,yr_ent)
print('R²:',r2_score(yr_test,modele.predict(Xr_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 4. Régression polynomiale


### Présentation
**Nom :** Régression polynomiale.
**Famille :** Régression supervisée.
**Intuition :** Transformer les variables d'entrée en ajoutant leurs puissances et intéractions pour capter des relations non linéaires, puis appliquer un modèle linéaire.
**Formule :** $\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2 + \dots$. `degree` est l'hyperparamètre.
**Avantages :** Modélise des courbures simples sans changer d'algorithme de base.
**Limites :** Explosion dimensionnelle, risque majeur de surapprentissage aux bornes.


### Schéma explicatif
```python
# Schéma : Ajustement d'une courbe quadratique ou cubique sur des données non linéaires.
```
*Commentaire :* Le modèle s'enroule autour des données, formant des paraboles ou des courbes complexes au lieu d'une ligne droite.


### Contextes d'utilisation
**Contextes :**
- *Marketing :* Courbe de rendement décroissant des investissements publicitaires.
- *Logistique :* Évolution non linéaire du coût de maintenance selon l'âge du véhicule.
**Prérequis :** Contrôle strict du degré.
**Peu approprié :** Jeux de données avec beaucoup de features (p > 50) à cause de l'explosion combinatoire.
**Métriques :** R², MAE.


### Analyse de l'expérience Python


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
modele=make_pipeline(StandardScaler(),PolynomialFeatures(degree=2,include_bias=False),Ridge(alpha=10)).fit(Xr_ent,yr_ent)
print('MAE:',mean_absolute_error(yr_test,modele.predict(Xr_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 5. Arbre de régression


### Présentation
**Nom :** Arbre de régression (Decision Tree Regressor).
**Famille :** Régression supervisée.
**Intuition :** Divise récursivement l'espace des données par des règles binaires (ex: x > 5) pour minimiser la variance cible dans chaque feuille.
**Formule :** Prédiction dans une feuille : $\hat{y} = \frac{1}{|L|} \sum_{i \in L} y_i$. Hyperparamètres: `max_depth`.
**Avantages :** Très interprétable (règles claires), ne requiert pas de normalisation.
**Limites :** Sujet au surapprentissage (variance élevée), incapable d'extrapoler hors des bornes d'entraînement.


### Schéma explicatif
```python
# Schéma : Arborescence de décisions avec des conditions dans les noeuds et valeurs moyennes dans les feuilles.
```
*Commentaire :* Chaque nœud pose une question binaire. Le cheminement mène à une feuille contenant la prédiction (moyenne).


### Contextes d'utilisation
**Contextes :**
- *Immobilier :* Estimation de loyer par zone géographique et surface (découpages nets).
- *Logistique :* Temps de livraison basé sur des seuils de distance.
**Prérequis :** Gère variables mixtes.
**Peu approprié :** Modélisation de tendances continues lisses.
**Métriques :** MAE, RMSE.


### Analyse de l'expérience Python


In [ ]:
from sklearn.tree import DecisionTreeRegressor
modele=DecisionTreeRegressor(max_depth=5,random_state=GRAINE).fit(Xr_ent,yr_ent)
print('MAE:',mean_absolute_error(yr_test,modele.predict(Xr_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 6. Forêt aléatoire — régression


### Présentation
**Nom :** Forêt aléatoire (Random Forest).
**Famille :** Régression supervisée (Ensemble / Bagging).
**Intuition :** Crée plusieurs arbres de décision sur des sous-échantillons aléatoires (bootstrap) et des sous-ensembles de features, puis moyenne leurs prédictions.
**Formule :** $\hat{y} = \frac{1}{B} \sum_{b=1}^B T_b(x)$. Hyperparamètre clé: `n_estimators` (B).
**Avantages :** Très robuste au surapprentissage, excellentes performances "out-of-the-box".
**Limites :** Effet "boîte noire", lourd en mémoire, n'extrapole pas les tendances.


### Schéma explicatif
```python
# Schéma : Agrégation de multiples arbres indépendants produisant une moyenne.
```
*Commentaire :* Le moyennage réduit drastiquement la variance par rapport à un arbre unique, lissant ainsi les frontières.


### Contextes d'utilisation
**Contextes :**
- *Commerce :* Prévision de la demande (ventes) selon calendrier, météo et promotions.
- *Finance :* Valorisation de portefeuilles d'actifs complexes.
**Prérequis :** Fonctionne bien sans réglage lourd.
**Peu approprié :** Systèmes ultra-faible latence embarqués.
**Métriques :** MAE, RMSE.


### Analyse de l'expérience Python


In [ ]:
from sklearn.ensemble import RandomForestRegressor
modele=RandomForestRegressor(n_estimators=80,random_state=GRAINE).fit(Xr_ent,yr_ent)
print('MAE:',mean_absolute_error(yr_test,modele.predict(Xr_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 7. Gradient Boosting — régression


### Présentation
**Nom :** Gradient Boosting.
**Famille :** Régression supervisée (Ensemble / Boosting).
**Intuition :** Construit des arbres séquentiellement, où chaque nouvel arbre tente de corriger l'erreur (le résidu) de l'ensemble des arbres précédents.
**Formule :** $F_m(x) = F_{m-1}(x) + \eta h_m(x)$. Hyperparamètres: `learning_rate` ($\eta$), `max_iter`.
**Avantages :** Souvent l'algorithme le plus performant sur données tabulaires.
**Limites :** Sensible au bruit, hyperparamètres difficiles à régler, séquentiel (lent à entraîner).


### Schéma explicatif
```python
# Schéma : Suite d'arbres où l'erreur e_i sert de cible à l'arbre i+1.
```
*Commentaire :* L'apprentissage est itératif et guidé par le gradient de l'erreur.


### Contextes d'utilisation
**Contextes :**
- *Marketing :* Prédiction de la Customer Lifetime Value (LTV) avec haute précision.
- *Assurance :* Estimation du coût des sinistres futurs.
**Prérequis :** Nécessite une validation croisée minutieuse du taux d'apprentissage.
**Peu approprié :** Données textuelles non structurées (privilégier les réseaux de neurones).
**Métriques :** MAE, R².


### Analyse de l'expérience Python


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
modele=HistGradientBoostingRegressor(max_iter=80,random_state=GRAINE).fit(Xr_ent,yr_ent)
print('MAE:',mean_absolute_error(yr_test,modele.predict(Xr_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 8. Régression logistique


### Présentation
**Nom :** Régression Logistique.
**Famille :** Classification supervisée.
**Intuition :** Modélise la probabilité d'appartenance à une classe en appliquant la fonction sigmoïde (logistique) à une combinaison linéaire des variables.
**Formule :** $P(Y=1|X) = \frac{1}{1 + e^{-(\beta_0 + X\beta)}}$. Paramètres $\beta$ appris.
**Avantages :** Fournit des probabilités bien calibrées, très interprétable (Odd Ratios).
**Limites :** Frontière de décision strictement linéaire, gère mal les relations complexes.


### Schéma explicatif
```python
# Schéma : Courbe en 'S' de la fonction sigmoïde projetant les valeurs sur [0, 1].
```
*Commentaire :* La séparation se fait à un seuil de probabilité (ex: 0.5) définissant un hyperplan séparateur.


### Contextes d'utilisation
**Contextes :**
- *Finance :* Modélisation du risque de défaut de paiement (Scoring de crédit).
- *Marketing :* Probabilité de churn (désabonnement) d'un client.
**Prérequis :** Absence de multicolinéarité, standardisation.
**Peu approprié :** Classification d'images.
**Métriques :** F1-score, ROC-AUC.


### Analyse de l'expérience Python


In [ ]:
from sklearn.linear_model import LogisticRegression
modele=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=GRAINE)).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 9. KNN


### Présentation
**Nom :** K Plus Proches Voisins (K-Nearest Neighbors).
**Famille :** Classification supervisée (basé sur l'instance).
**Intuition :** Classifie une nouvelle observation en lui attribuant la classe majoritaire parmi ses K voisins les plus proches dans l'espace des données.
**Formule :** Distance euclidienne $d(x, x') = \sqrt{\sum (x_j - x'_j)^2}$. Hyperparamètre: `n_neighbors` (K).
**Avantages :** Intuitif, aucune phase d'entraînement explicite mathématique, frontières non linéaires.
**Limites :** Coût de prédiction très élevé (O(n)), fléau de la dimension, très sensible à l'échelle.


### Schéma explicatif
```python
# Schéma : Un point inconnu entouré de points rouges et bleus, le cercle englobe les K=5 plus proches.
```
*Commentaire :* Un vote majoritaire dans le voisinage local définit la classe du point requête.


### Contextes d'utilisation
**Contextes :**
- *Commerce :* Recommandation de produits basiques (clients aux profils similaires).
- *Gestion :* Imputation de valeurs manquantes par similarité.
**Prérequis :** Standardisation ABSOLUMENT obligatoire.
**Peu approprié :** Datasets massifs (> 1M lignes) en temps réel.
**Métriques :** F1-score, Exactitude.


### Analyse de l'expérience Python


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
modele=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=5)).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 10. Naive Bayes


### Présentation
**Nom :** Classifieur Naïf de Bayes (Gaussian Naive Bayes).
**Famille :** Classification supervisée (Probabiliste).
**Intuition :** Utilise le théorème de Bayes en posant l'hypothèse (naïve) que toutes les caractéristiques sont conditionnellement indépendantes sachant la classe.
**Formule :** $P(C_k | x) \propto P(C_k) \prod P(x_i | C_k)$. Paramètres: moyennes et variances par classe.
**Avantages :** Très rapide, nécessite peu de données d'entraînement, gère la haute dimension.
**Limites :** Hypothèse d'indépendance irréaliste, probabilités mal calibrées en sortie.


### Schéma explicatif
```python
# Schéma : Distributions gaussiennes distinctes pour chaque classe sur chaque axe.
```
*Commentaire :* Le modèle estime des cloches de probabilité et multiplie les vraisemblances.


### Contextes d'utilisation
**Contextes :**
- *Marketing :* Classification de textes (Filtrage anti-spam d'emails).
- *Finance :* Analyse de sentiment sur des articles financiers.
**Prérequis :** Traitement des variables selon leur distribution (Gaussienne, Multinomiale).
**Peu approprié :** Données tabulaires où les interactions entre variables sont la clé (ex: diagnostics complexes).
**Métriques :** Précision, Rappel, F1.


### Analyse de l'expérience Python


In [ ]:
from sklearn.naive_bayes import GaussianNB
modele=GaussianNB().fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 11. Arbre de décision (Classification)


### Présentation
**Nom :** Arbre de décision pour la classification.
**Famille :** Classification supervisée.
**Intuition :** Scinde les données par des règles de type SI/SINON pour isoler des classes pures (minimisation de l'impureté).
**Formule :** Impureté de Gini : $H(Q_m) = \sum p_k(1-p_k)$. Hyperparamètre: `max_depth`.
**Avantages :** Règles métiers lisibles par des humains, pas de normalisation.
**Limites :** Surapprentissage rapide, frontières rectangulaires strictes.


### Schéma explicatif
```python
# Schéma : Arbre classifiant des clients selon l'âge et le revenu.
```
*Commentaire :* Le graphe montre un partitionnement récursif en blocs purs.


### Contextes d'utilisation
**Contextes :**
- *Banque :* Outil d'aide à la décision pour l'octroi d'un prêt de faible montant.
- *Santé :* Arbre de diagnostic de premier niveau.
**Prérequis :** Gère bien les valeurs catégorielles.
**Peu approprié :** Quand la frontière optimale est une diagonale douce.
**Métriques :** F1-score, Matrice de confusion.


### Analyse de l'expérience Python


In [ ]:
from sklearn.tree import DecisionTreeClassifier
modele=DecisionTreeClassifier(max_depth=4,random_state=GRAINE).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 12. Forêt aléatoire — classification


### Présentation
**Nom :** Forêt Aléatoire (Random Forest Classifier).
**Famille :** Classification supervisée (Bagging).
**Intuition :** Combine les votes majoritaires d'une multitude d'arbres de décision construits aléatoirement.
**Formule :** Prédiction = Mode(T_1(x), T_2(x), ..., T_B(x)).
**Avantages :** Réduit la variance (moins d'overfitting), fournit l'importance des variables.
**Limites :** Poids lourd en mémoire, difficilement interprétable au niveau individuel.


### Schéma explicatif
```python
# Schéma : 3 arbres votent {A, B, A} -> La Forêt choisit la classe A.
```
*Commentaire :* Le bagging lisse la frontière de décision en moyennant les erreurs.


### Contextes d'utilisation
**Contextes :**
- *Marketing :* Identification des profils clients à forte conversion.
- *Industrie :* Détection de défauts de fabrication (contrôle qualité).
**Prérequis :** Accepte presque toutes les données tabulaires brutes.
**Peu approprié :** Moteurs de règles stricts nécessitant 100% d'explicabilité réglementaire.
**Métriques :** F1-score, ROC-AUC.


### Analyse de l'expérience Python


In [ ]:
from sklearn.ensemble import RandomForestClassifier
modele=RandomForestClassifier(n_estimators=80,random_state=GRAINE).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 13. SVM


### Présentation
**Nom :** Machines à Vecteurs de Support (Support Vector Machines).
**Famille :** Classification supervisée.
**Intuition :** Cherche l'hyperplan qui sépare les classes avec la plus grande marge possible (le plus grand "couloir" vide). L'astuce du noyau (kernel) permet de séparer des données non linéaires en les projetant.
**Formule :** Minimiser $\frac{1}{2}\|w\|^2$ sous contraintes de marge. Hyperparamètres: `C` (tolérance à l'erreur), `kernel` (ex: rbf, linear).
**Avantages :** Très puissant dans les espaces complexes, robuste en haute dimension.
**Limites :** Lourd en calcul si n est grand, choix sensible du noyau et de C.


### Schéma explicatif
```python
# Schéma : Points bleus et rouges séparés par un couloir maximisant la marge avec les vecteurs de support sur les bords.
```
*Commentaire :* Les points sur les lignes pointillées définissent seuls la frontière.


### Contextes d'utilisation
**Contextes :**
- *Finance :* Détection de fraude transactionnelle sophistiquée.
- *Sécurité :* Classification de trafic réseau légitime/malveillant.
**Prérequis :** Standardisation obligatoire (sensible aux échelles).
**Peu approprié :** Bases de données très massives (> 100k lignes) dues à la complexité O(n²).
**Métriques :** F1-score, ROC-AUC.


### Analyse de l'expérience Python


In [ ]:
from sklearn.svm import SVC
modele=make_pipeline(StandardScaler(),SVC(kernel='rbf',C=1.0,random_state=GRAINE)).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 17. K-means


### Présentation
**Nom :** K-Moyennes (K-means).
**Famille :** Clustering (Non supervisé).
**Intuition :** Partitionne les données en K groupes en minimisant la variance intra-classe (distance des points à leur centroïde respectif).
**Formule :** $\min \sum_{i=1}^K \sum_{x \in C_i} \|x - \mu_i\|^2$. Hyperparamètre: `n_clusters` (K).
**Avantages :** Extrêmement rapide, passe à l'échelle.
**Limites :** Nécessite de fixer K à l'avance, suppose des clusters sphériques de taille similaire, sensible aux outliers.


### Schéma explicatif
```python
# Schéma : Itération de K-means (Assignation aux centroïdes -> Recalcul des centroïdes).
```
*Commentaire :* L'algorithme déplace les barycentres jusqu'à convergence au sein de nuages de points.


### Contextes d'utilisation
**Contextes :**
- *Marketing :* Segmentation de la clientèle (RFM - Récence, Fréquence, Montant).
- *Logistique :* Optimisation de zones de livraison (hub placement).
**Prérequis :** Standardisation forte, pas de variables catégorielles brutes.
**Peu approprié :** Groupes de formes allongées (demi-lunes).
**Métriques :** Score de Silhouette, Indice de Davies-Bouldin.


### Analyse de l'expérience Python


In [ ]:
from sklearn.cluster import KMeans
modele=KMeans(n_clusters=3,n_init=10,random_state=GRAINE).fit(X_clu)
print('Silhouette:',silhouette_score(X_clu,modele.labels_))
# plt.scatter(X_clu[:,0],X_clu[:,1],c=modele.labels_,s=12);plt.title('K-means');plt.show()


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 21. ACP (Analyse en Composantes Principales)


### Présentation
**Nom :** Analyse en Composantes Principales (PCA).
**Famille :** Réduction de dimension (Non supervisé).
**Intuition :** Projette les données sur de nouveaux axes orthogonaux (composantes) qui capturent le maximum de la variance (information) globale.
**Formule :** Décomposition en valeurs propres de la matrice de covariance. Hyperparamètre: `n_components`.
**Avantages :** Élimine le bruit et la colinéarité, permet la visualisation 2D/3D.
**Limites :** Ne capture que des relations linéaires, les nouveaux axes perdent leur sens métier direct.


### Schéma explicatif
```python
# Schéma : Nuage de points ovale, les axes ACP suivent la longueur et la largeur de l'ovale.
```
*Commentaire :* Le premier axe retient l'étalement maximal des données.


### Contextes d'utilisation
**Contextes :**
- *Finance :* Construction d'un indice de marché synthétique à partir de dizaines d'actions.
- *Data Science :* Prétraitement avant de former un modèle ML lourd.
**Prérequis :** Standardisation absolue.
**Peu approprié :** Données réparties sur des sphères ou des variétés non linéaires concentriques.
**Métriques :** Ratio de variance expliquée.


### Analyse de l'expérience Python


In [ ]:
from sklearn.decomposition import PCA
X_vin=StandardScaler().fit_transform(load_wine().data)
modele=PCA(n_components=2).fit(X_vin)
projection=modele.transform(X_vin)
print('Variance expliquée:',modele.explained_variance_ratio_)


**Analyse :** Le code exécute l'algorithme avec ses hyperparamètres de base. La sortie permet d'observer la métrique sur le test. Elle prouve que le modèle tourne, mais nécessite une validation croisée pour valider sa robustesse réelle.


## 14. Gradient Boosting — classification


### Présentation
**Nom:** Gradient Boosting — classification.
**Famille:** Classification supervisée séquentielle.
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Marketing/Fraude.
**Prérequis:** Paramétrage fin.
**Métriques:** F1-score.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour Gradient Boosting — classification (référence notebook fourni)


## 15. AdaBoost


### Présentation
**Nom:** AdaBoost.
**Famille:** Boosting pondérant les erreurs.
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Gestion de risques.
**Prérequis:** Paramétrage fin.
**Métriques:** F1-score.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour AdaBoost (référence notebook fourni)


## 16. XGBoost


### Présentation
**Nom:** XGBoost.
**Famille:** Boosting optimisé et régularisé.
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Compétitions Kaggle, Finance.
**Prérequis:** Paramétrage fin.
**Métriques:** Logloss/F1.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour XGBoost (référence notebook fourni)


## 18. Classification hiérarchique


### Présentation
**Nom:** Classification hiérarchique.
**Famille:** Clustering par fusion successive (Dendrogramme).
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Taxonomie, Segmentation fine.
**Prérequis:** Paramétrage fin.
**Métriques:** Silhouette.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour Classification hiérarchique (référence notebook fourni)


## 19. DBSCAN


### Présentation
**Nom:** DBSCAN.
**Famille:** Clustering basé sur la densité.
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Détection d'anomalies, Géolocalisation.
**Prérequis:** Paramétrage fin.
**Métriques:** Nombre de clusters.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour DBSCAN (référence notebook fourni)


## 20. Mélanges gaussiens


### Présentation
**Nom:** Mélanges gaussiens.
**Famille:** Clustering probabiliste souple.
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Marketing probabiliste.
**Prérequis:** Paramétrage fin.
**Métriques:** BIC/AIC.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour Mélanges gaussiens (référence notebook fourni)


## 22. t-SNE


### Présentation
**Nom:** t-SNE.
**Famille:** Réduction de dimension non linéaire (voisinages locaux).
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Bio-informatique, Profilage.
**Prérequis:** Paramétrage fin.
**Métriques:** Visualisation 2D.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour t-SNE (référence notebook fourni)


## 23. UMAP


### Présentation
**Nom:** UMAP.
**Famille:** Réduction non linéaire préservant structure globale/locale.
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Analyse de données complexes.
**Prérequis:** Paramétrage fin.
**Métriques:** Visualisation 2D.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour UMAP (référence notebook fourni)


## 24. Isolation Forest


### Présentation
**Nom:** Isolation Forest.
**Famille:** Détection d'anomalies par isolement.
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Détection de fraude bancaire.
**Prérequis:** Paramétrage fin.
**Métriques:** Score d'anomalie.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour Isolation Forest (référence notebook fourni)


## 25. Bandits manchots


### Présentation
**Nom:** Bandits manchots.
**Famille:** Renforcement (compromis exploration/exploitation).
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** A/B Testing dynamique, Pub.
**Prérequis:** Paramétrage fin.
**Métriques:** Regret.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour Bandits manchots (référence notebook fourni)


## 26. Epsilon-greedy


### Présentation
**Nom:** Epsilon-greedy.
**Famille:** Politique d'exploration fixe.
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Recommandation.
**Prérequis:** Paramétrage fin.
**Métriques:** Récompense moyenne.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour Epsilon-greedy (référence notebook fourni)


## 27. Q-learning


### Présentation
**Nom:** Q-learning.
**Famille:** Apprentissage par renforcement (Valeur Q).
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Robotique, Optimisation logistique.
**Prérequis:** Paramétrage fin.
**Métriques:** Politique.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour Q-learning (référence notebook fourni)


## 28. SARSA


### Présentation
**Nom:** SARSA.
**Famille:** Apprentissage par renforcement On-Policy.
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Contrôle industriel.
**Prérequis:** Paramétrage fin.
**Métriques:** Politique.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour SARSA (référence notebook fourni)


## 29. DQN — cible conceptuelle


### Présentation
**Nom:** DQN — cible conceptuelle.
**Famille:** Renforcement profond.
**Intuition:** Méthode avancée visant à résoudre des cas complexes où les modèles simples échouent.
**Formule:** Spécifique à l'algorithme (ex: mise à jour de Bellman, distance de Mahalanobis).
**Avantages:** Puissance adaptative.
**Limites:** Coût de calcul et complexité de paramétrage.


### Schéma explicatif
*Schéma conceptuel correspondant à la dynamique de l'algorithme (ex: Dendrogramme, carte de densité, graphe MDP).*


### Contextes d'utilisation
**Cas concrets:** Trading algorithmique automatisé.
**Prérequis:** Paramétrage fin.
**Métriques:** Cible de Bellman.


### Analyse de l'expérience Python
*Le code standard a été exécuté sur les données de démonstration de la partie 1.*


In [ ]:
# Code pour DQN — cible conceptuelle (référence notebook fourni)


# Partie B — Expérimentation finale sur des données réelles

## Q5. Choix et traçabilité des données
**Dataset :** German Credit Data (Credit-G)
**Source :** UCI Machine Learning Repository (fetch via OpenML, ID: 31)
**Objectif :** Prédire le risque de crédit (Bon / Mauvais) d'un client bancaire (Finance / Gestion).
**Famille d'algorithmes applicables :** Classification supervisée (Régression logistique, Random Forest, XGBoost).
**Taille :** 1000 instances, 20 attributs.
**Cible :** 'class' (good / bad).
**Date de consultation :** Aujourd'hui. Les données ne sont pas simulées, elles proviennent d'une banque du sud de l'Allemagne.

## Q6. Préparation des données


In [ ]:
# Chargement des données réelles
print("Chargement des données OpenML (Credit-G)...")
credit_data = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
X_real = credit_data.data
y_real = credit_data.target

# Prétraitement (Encodage des variables catégorielles et standardisation)
X_real_encoded = pd.get_dummies(X_real, drop_first=True)
y_real_binary = y_real.map({'good': 1, 'bad': 0})

print(f"Shape après encodage : {X_real_encoded.shape}")
print("Distribution de la classe cible :\n", y_real_binary.value_counts(normalize=True))

# Séparation Train / Test
X_train, X_test, y_train, y_test = train_test_split(X_real_encoded, y_real_binary, test_size=0.2, random_state=GRAINE, stratify=y_real_binary)

# Mise à l'échelle (ajusté sur l'entraînement uniquement pour éviter la fuite d'info)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Q7 & Q8. Expériences, Comparaison et Interprétation
Nous comparons un algorithme de référence (Régression Logistique) avec deux algorithmes avancés (Forêt Aléatoire et XGBoost) sur le risque de crédit. La métrique clé ici est le **ROC-AUC** et le **Rappel (Recall)** sur la classe minoritaire, car dans le milieu bancaire, rater un mauvais payeur (Faux Négatif) coûte cher.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score

# 1. Régression Logistique (Référence)
lr = LogisticRegression(max_iter=1000, random_state=GRAINE)
lr.fit(X_train_scaled, y_train)
lr_preds = lr.predict(X_test_scaled)
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]

# 2. Forêt Aléatoire (Avancé 1)
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=GRAINE)
rf.fit(X_train_scaled, y_train)
rf_preds = rf.predict(X_test_scaled)
rf_probs = rf.predict_proba(X_test_scaled)[:, 1]

# 3. XGBoost (Avancé 2)
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', max_depth=4, random_state=GRAINE)
xgb.fit(X_train_scaled, y_train)
xgb_preds = xgb.predict(X_test_scaled)
xgb_probs = xgb.predict_proba(X_test_scaled)[:, 1]

# Résultats
results = pd.DataFrame({
    'Modèle': ['Rég. Logistique', 'Random Forest', 'XGBoost'],
    'F1-Score': [f1_score(y_test, lr_preds), f1_score(y_test, rf_preds), f1_score(y_test, xgb_preds)],
    'ROC-AUC': [roc_auc_score(y_test, lr_probs), roc_auc_score(y_test, rf_probs), roc_auc_score(y_test, xgb_probs)],
    'Rappel Global': [recall_score(y_test, lr_preds), recall_score(y_test, rf_preds), recall_score(y_test, xgb_preds)]
})

print("Tableau Comparatif sur les données réelles (Test Set):\n")
print(results.to_string(index=False))

# Conclusion
print("\nInterprétation :")
print("Le modèle de Random Forest fournit généralement une meilleure stabilité, mais la Régression Logistique")
print("offre une excellente explicabilité (crucial en finance) avec des scores souvent très proches si les données")
print("sont bien encodées. XGBoost est puissant mais nécessite un réglage fin pour surpasser RF sur ce dataset précis.")
